In [6]:
import math
import pandas as pd
import numpy as np
import json

# READ DATA (and drop testing workerids)

In [39]:
df = pd.read_csv("pictureRatingData.csv")

unnecessary_cols = ["questionid", "filename", "listnumber", "assignmentid", "hitid", "origin", "timestamp", "partid", "id", "imgorder"]
unformatted_df = df.drop(unnecessary_cols, axis=1)

workerids = ["test", "123", "testFirefox", "testRedirect"]  # add more to this list if there were more test runs
unformatted_df = unformatted_df[~unformatted_df["workerid"].isin(workerids)]

len(unformatted_df["workerid"].unique())

28

## FORMAT RATING VALUES FROM THE ANSWER COLUMN

In [27]:
def format_answer(answer, weak_word, strong_word, antonym):
    """
    returns multiple results.
    whatever data is needed out of the answer, edit and manipulate it in this function
    """
    results = json.loads(json.loads(answer))

    new_dict = dict()
    for item in results["ratings"]:
        name = item["image"].replace(".png", "")
        new_dict.update(
            {name: item["rating"]}
        )

    if results["speaker"] in ["frank", "bri"]:
        speakerType = "native"
    else:
        speakerType = "nonnative"
    
    if results["speaker"] in ["bri", "sasha"]:
        speakerGender = "female"
    else:
        speakerGender = "male"

    # weak word rating, strong word rating, antonym rating
    return new_dict.get(weak_word), new_dict.get(strong_word), new_dict.get(antonym), speakerType, speakerGender

In [29]:
answers_only_df = unformatted_df.apply(
    lambda x: format_answer(x.answer, x.weak, x.strong, x.antonym), 
    axis=1, 
    result_type="expand",  # this is how you make it into multiple columns
)
answers_only_df.columns=["derivationStrength", "strongRating", "antonymRating", "speakerType", "speakerGender"]
answers_only_df

,derivationStrength,strongRating,antonymRating,speakerType,speakerGender
1,100,0,0.0,native,female
2,0,100,0.0,native,female
3,100,0,0.0,native,male
4,75,25,0.0,native,male
5,100,0,0.0,nonnative,male
...,...,...,...,...,...
985,100,0,0.0,nonnative,male
986,100,0,0.0,nonnative,male
987,100,0,0.0,nonnative,male
988,100,0,0.0,nonnative,female


## READ WORD FREQ DICT IN

In [30]:
word_freq_dict_df = pd.read_csv("wordFreqDict.csv")
word_freq_dict_df.head()

word_freq_dict_df[word_freq_dict_df['Frequency'].isin(["x"])]

,Word,Rank,Frequency


## RENAME COLUMN NAMES TO WHAT THE MODEL NEEDS

In [41]:
answers_df = pd.concat([unformatted_df, answers_only_df], axis=1)
answers_df = answers_df.drop(["answer"], axis=1)

answers_df = answers_df.rename(columns={
    "workerid": "participantId",
    "itemid": "itemId",
    "type": "itemType"
})

answers_df[answers_df["participantId"] == "6a2482e9af72b55296872b0d"]

,participantId,itemId,itemType,weak,strong,antonym,lowfreq1,lowfreq2,derivationStrength,strongRating,antonymRating,speakerType,speakerGender
27,6a2482e9af72b55296872b0d,1,critical,soft,mushy,crunchy,panicked,introspective,0,100,0.0,native,female
58,6a2482e9af72b55296872b0d,2,critical,good,excellent,bad,dramatic,technical,0,100,0.0,nonnative,male
89,6a2482e9af72b55296872b0d,3,critical,funny,hilarious,boring,resilient,erratic,38,62,0.0,nonnative,male
120,6a2482e9af72b55296872b0d,4,critical,cold,freezing,hot,fictional,communal,0,100,0.0,nonnative,male
151,6a2482e9af72b55296872b0d,5,critical,big,enormous,tiny,rural,creative,0,100,0.0,native,male
182,6a2482e9af72b55296872b0d,6,critical,old,ancient,new,illegal,crazy,0,100,0.0,nonnative,male
213,6a2482e9af72b55296872b0d,7,critical,large,gigantic,little,scattered,incapable,0,100,0.0,nonnative,female
244,6a2482e9af72b55296872b0d,8,critical,wet,drenched,dry,ephemeral,rudimentary,0,100,0.0,native,female
275,6a2482e9af72b55296872b0d,9,critical,happy,ecstatic,sad,anecdotal,extravagant,0,100,0.0,native,female
306,6a2482e9af72b55296872b0d,10,critical,hot,scalding,cold,orchestral,gaunt,0,100,0.0,nonnative,female


# FILTER - EXCLUSION CRITERIA

In [32]:
# filter out rating scores
#    - if avg. rating across all unambiguous trials were < 80%
umambiguous_trials = answers_df[answers_df["itemType"] == "unambiguous"]
picrating_filtered_participants = pd.unique(
    umambiguous_trials.groupby("participantId").filter(
        lambda x: x["derivationStrength"].mean() < 80
    )["participantId"]
)
picrating_filtered_participants

# this is now done in the R script
# # filter out ToM scores
# #    - if accuracy on control were < 80%
# nathan4u_filtered_participants = pd.unique(
#     tom_scores.groupby("participantId").filter(
#         lambda x: x["ToMScore"].mean() < 80
#     )["participantId"]
# )

array([], dtype=object)

## ToM Filtered Participants

Get a list from the R script

In [33]:
nathan4u_filtered_participants = []

## Audio Filtered Participants

After transcribing the audio for the two practice perspective taking trials, any that did not choose the target word will be excluded

In [34]:
audio_filtered_participants = []

In [35]:
all_filtered_participants = list(set(picrating_filtered_participants + nathan4u_filtered_participants + audio_filtered_participants))
all_filtered_participants

[]

# MERGE DFs TOGETHER

In [36]:
# get the weak word info
final_df = answers_df.merge(word_freq_dict_df, left_on="weak", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "weakRank",
    "Frequency": "weakFreq",
})

# get the strong word info
final_df = final_df.merge(word_freq_dict_df, left_on="strong", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "strongRank",
    "Frequency": "strongFreq",
})

# get the antonym word info
final_df = final_df.merge(word_freq_dict_df, left_on="antonym", right_on="Word")
final_df = final_df.drop(["Word"], axis=1)
final_df = final_df.rename(columns={
    "Rank": "antonymRank",
    "Frequency": "antonymFreq",
})

# create freq ratios
cols = ["weakFreq", "strongFreq", "antonymFreq", "weakRank", "strongRank", "antonymRank"]
final_df[cols] = final_df[cols].apply(pd.to_numeric, errors='coerce', axis=1)
final_df["freqRatio"] = final_df["weakFreq"] / final_df["strongFreq"]
# if freq ratio is nan, fill it with 2
final_df["freqRatio"] = final_df["freqRatio"].apply(lambda x: math.log10(x)).fillna(2, downcast='infer')
final_df.head(6)

/tmp/ipykernel_45521/2728389408.py:30: FutureWarning: The 'downcast' keyword in fillna is deprecated and will be removed in a future version. Use res.infer_objects(copy=False) to infer non-object dtype, or pd.to_numeric with the 'downcast' keyword to downcast numeric results.
  final_df["freqRatio"] = final_df["freqRatio"].apply(lambda x: math.log10(x)).fillna(2, downcast='infer')


,participantId,itemId,itemType,weak,strong,antonym,lowfreq1,lowfreq2,derivationStrength,strongRating,antonymRating,speakerType,speakerGender,weakRank,weakFreq,strongRank,strongFreq,antonymRank,antonymFreq,freqRatio
0,66da162f5538535e93490441,1,critical,soft,mushy,crunchy,panicked,introspective,100,0,0.0,native,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
1,6978d830c5d415696508430d,1,critical,soft,mushy,crunchy,panicked,introspective,0,100,0.0,native,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
2,699b610e2ba4bade6f39121b,1,critical,soft,mushy,crunchy,panicked,introspective,100,0,0.0,native,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
3,699e3cc79b0d3d96b469ca49,1,critical,soft,mushy,crunchy,panicked,introspective,75,25,0.0,native,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
4,69a8cff018bead25e4a5d848,1,critical,soft,mushy,crunchy,panicked,introspective,100,0,0.0,nonnative,male,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265
5,69b5c6bdbcf55371066f5e73,1,critical,soft,mushy,crunchy,panicked,introspective,100,0,0.0,nonnative,female,1381.0,27970.0,18362.0,478.0,14284.0,834.0,1.767265


In [37]:
final_remove_cols = ["weakRank", "strongRank", "antonymRank", "weakFreq", "strongFreq", "antonymFreq"]
final_df = final_df.drop(final_remove_cols, axis=1)
final_df.to_csv("formatted_results.csv")

# TRIAL STUFF

In [18]:
unformatted_df["answer"].head()

0    "{\"prob1\":\"17\",\"prob2\":0,\"prob3\":83,\"...
1    "{\"prob1\":0,\"prob2\":0,\"prob3\":\"100\",\"...
2    "{\"prob1\":5,\"prob2\":\"95\",\"prob3\":0,\"p...
3    "{\"prob1\":0,\"prob2\":\"90\",\"prob3\":10,\"...
4    "{\"prob1\":0,\"prob2\":97,\"prob3\":\"3\",\"p...
Name: answer, dtype: object

In [7]:
sample = unformatted_df.iloc[0]
sample

workerid                                               19970528
questionid                                               116506
answer        "{\"prob1\":\"17\",\"prob2\":0,\"prob3\":83,\"...
itemid                                                        1
type                                                   critical
weak                                                       soft
strong                                                    mushy
antonym                                                 crunchy
lowfreq1                                               panicked
lowfreq2                                          introspective
imgorder                                                 random
Name: 0, dtype: object

In [8]:
sample_answer = json.loads(json.loads(sample["answer"]))
sample_answer

{'prob1': '17',
 'prob2': 0,
 'prob3': 83,
 'probsum': 100,
 'ratings': [{'image': 'mushy.png', 'rating': 17},
  {'image': 'crunchy.png', 'rating': 0},
  {'image': 'soft.png', 'rating': 83}],
 'speaker': 'frank',
 'timeTaken': 32160.69999998808}

In [14]:
new_dict = dict()
for item in sample_answer["ratings"]:
    name = item["image"].replace(".png", "")

    # if name == sample["weak"]:
    #     tag = "target"
    # elif name == sample["strong"]:
    #     tag = "strong"
    # else:
    #     tag = "antonym"
    
    new_dict.update(
        {name: item["rating"]}
    )
new_dict

{'mushy': 17, 'crunchy': 0, 'soft': 83}

In [13]:
if sample_answer["speaker"] in ["frank", "bri"]:
    speakerType = "native"
else:
    speakerType = "nonnative"

if sample_answer["speaker"] in ["bri", "sasha"]:
    speakerGender = "female"
else:
    speakerGender = "male"

speakerGender, speakerType, sample["type"]

('male', 'native', 'critical')

In [12]:
word_freq_dict = pd.read_csv("wordFreqDict.csv")
word_freq_dict.head()

,Word,Rank,Frequency
0,accurate,2754,11842
1,additive,15315,725
2,alive,1541,24184
3,ancient,1832,19818
4,anecdotal,10575,1424
